# Fink/LSST — Select DIA Object → (visitId, detector) list for pipetask

This notebook loads the visit index files produced by `01_fink_block_lightcurves.ipynb`
and provides tools to:

1. Visualise the temporal and filter sampling of all DIA objects
2. Select a specific `diaObjectId` (from diaSources and/or forced photometry)
3. Extract the corresponding list of `(visit, detector)` tuples
4. Export a CSV ready to pass to `pipetask run` at the USDF

## Input files (produced by `01_fink_block_lightcurves.ipynb`)

| File | Description |
|------|-------------|
| `visit_index.csv` | Visit summary from diaSources — one row per (diaObjectId, visit, detector, band) |
| `visit_index_fp.csv` | Visit summary from forced photometry (if `r:visit` available in fp) |

Both files share the same schema:

| Column | Type | Description |
|--------|------|-------------|
| `diaObjectId` | int64 | Unique DIA object identifier |
| `group` | str | Fink classification group |
| `source_type` | str | `'src'` or `'fp'` |
| `visit` | Int64 | Visit ID (13-digit, e.g. `2026030900027`) |
| `detector` | Int64 | CCD detector number (0–188) |
| `band` | str | Photometric band |
| `mjd_first` | float | MJD of first detection in this visit |
| `n_det` | int | Number of detections in this visit |
| `x_mean` | float | Mean pixel x position |
| `y_mean` | float | Mean pixel y position |
| `xErr_mean` | float | Mean pixel x uncertainty |
| `yErr_mean` | float | Mean pixel y uncertainty |
| `psfFlux_mean` | float | Mean PSF flux (nJy) |

## Usage for `pipetask run` (USDF)

The output CSV from `get_visit_detector_list()` can be used directly to
build a Butler query and submit a DRP reconstruction job:

```bash
pipetask run \
    -i LSSTCam/default \
    -b /repo/main \
    -p ${DRP_PIPE_DIR}/pipelines/LSSTCam/DRP.yaml#stage1-single-visit,stage2-recalibrate \
    -o u/dagore/fink_alertes_run2026_01 \
    -d "visit IN (2026030900027, 2026030900031) AND detector IN (42, 43)" \
    --register-dataset-types
```


## 1. Imports & configuration

In [ ]:
import matplotlib.pyplot as plt
import pandas as pd
import numpy as np
import os
import glob
import warnings

warnings.filterwarnings("ignore")

print(f"pandas  : {pd.__version__}")
print(f"numpy   : {np.__version__}")

In [ ]:
# ── Paths ─────────────────────────────────────────────────────────────────────
NB_TAG = "FINK_BLOCK_LC_01"
DIR_DATA = f"data_{NB_TAG}"

# Paths to the visit index files produced by 01_fink_block_lightcurves.ipynb
VISIT_INDEX_SRC_PATH = os.path.join(DIR_DATA, "visit_index.csv")  # diaSources
VISIT_INDEX_FP_PATH = os.path.join(DIR_DATA, "visit_index_fp.csv")  # forced photometry

# Band → y-axis position for timeline plots
BAND_MAP = {"u": 0, "g": 1, "r": 2, "i": 3, "z": 4, "y": 5}
BAND_COLORS = {
    "u": "#9b59b6",
    "g": "#2ecc71",
    "r": "#e74c3c",
    "i": "#e67e22",
    "z": "#3498db",
    "y": "#795548",
}

plt.rcParams.update({"figure.dpi": 120, "axes.grid": True, "grid.alpha": 0.3, "font.size": 9})

print(f"Data directory : {os.path.abspath(DIR_DATA)}")
for p in (VISIT_INDEX_SRC_PATH, VISIT_INDEX_FP_PATH):
    status = "found" if os.path.exists(p) else "NOT FOUND"
    print(f"  {os.path.basename(p)}: {status}")

## 2. Load visit index files

In [ ]:
def load_visit_index(path: str) -> pd.DataFrame:
    """
    Load a visit index CSV produced by 01_fink_block_lightcurves.ipynb.

    Handles both the new schema (source_type, x_mean, xErr_mean, ...)
    and the legacy schema from 03_fink_add_visitId.ipynb (r:visit, r:detector, ...).

    Returns a normalised DataFrame with columns:
        diaObjectId, group, source_type, visit, detector, band,
        mjd_first, n_det, x_mean, y_mean, xErr_mean, yErr_mean, psfFlux_mean
    """
    if not os.path.exists(path):
        print(f"  File not found: {path}")
        return pd.DataFrame()

    df = pd.read_csv(path)

    # ── Legacy schema compatibility: rename r:* columns ───────────────────────
    rename_map = {}
    for old, new in [
        ("r:visit", "visit"),
        ("r:detector", "detector"),
        ("r:band", "band"),
        ("r:midpointMjdTai", "mjd_first"),
        ("r:x", "x_mean"),
        ("r:y", "y_mean"),
        ("r:xErr", "xErr_mean"),
        ("r:yErr", "yErr_mean"),
        ("r:psfFlux", "psfFlux_mean"),
    ]:
        if old in df.columns and new not in df.columns:
            rename_map[old] = new
    if rename_map:
        df = df.rename(columns=rename_map)

    # ── Ensure required columns exist ─────────────────────────────────────────
    for col in ("group", "source_type"):
        if col not in df.columns:
            df[col] = "unknown"
    for col in ("x_mean", "y_mean", "xErr_mean", "yErr_mean", "psfFlux_mean", "n_det", "mjd_first"):
        if col not in df.columns:
            df[col] = np.nan

    # ── Cast visit and detector to nullable integers ───────────────────────────
    for col in ("visit", "detector"):
        if col in df.columns:
            df[col] = pd.to_numeric(df[col], errors="coerce").astype("Int64")

    return df


# Load both sources
df_src = load_visit_index(VISIT_INDEX_SRC_PATH)
df_fp = load_visit_index(VISIT_INDEX_FP_PATH)

print("=== diaSources visit index ===")
if not df_src.empty:
    print(f"  Rows             : {len(df_src)}")
    print(f"  Unique objects   : {df_src['diaObjectId'].nunique()}")
    print(f"  Unique visits    : {df_src['visit'].dropna().nunique()}")
    print(f"  Groups           : {sorted(df_src['group'].unique())}")
    print(f"  Columns          : {list(df_src.columns)}")
    print(f"  xErr available   : {df_src['xErr_mean'].notna().any()}")
else:
    print("  Not available.")

print("\n=== Forced Photometry visit index ===")
if not df_fp.empty:
    print(f"  Rows             : {len(df_fp)}")
    print(f"  Unique objects   : {df_fp['diaObjectId'].nunique()}")
    print(f"  Unique visits    : {df_fp['visit'].dropna().nunique()}")
    print(f"  xErr available   : {df_fp['xErr_mean'].notna().any()}")
else:
    print("  Not available (r:visit may not be exposed for fp in this API version).")

## 3. Overview: all DIA objects timeline

In [ ]:
def plot_all_objects_timeline(
    df: pd.DataFrame,
    title: str = "DIA Objects — temporal sampling by band",
    max_objects: int = 80,
) -> None:
    """
    Plot the temporal sampling of all DIA objects on a band vs MJD grid.
    Each object gets a distinct colour; band is on the y-axis.

    Parameters
    ----------
    df : DataFrame with columns diaObjectId, mjd_first, band.
    title : Figure title.
    max_objects : Cap on distinct objects shown (for readability).
    """
    if df.empty or "mjd_first" not in df.columns:
        print("No data to plot.")
        return

    unique_objects = df["diaObjectId"].unique()
    n_obj = min(len(unique_objects), max_objects)
    unique_objects = unique_objects[:n_obj]

    cmap = plt.cm.get_cmap("tab20", n_obj)

    fig, ax = plt.subplots(figsize=(18, 5))

    for i, obj in enumerate(unique_objects):
        sub = df[df["diaObjectId"] == obj].dropna(subset=["mjd_first", "band"])
        y = [BAND_MAP.get(b, -1) for b in sub["band"]]
        ax.scatter(sub["mjd_first"], y, color=cmap(i), label=str(obj), alpha=0.7, s=18)

    ax.set_yticks(list(BAND_MAP.values()))
    ax.set_yticklabels(list(BAND_MAP.keys()))
    ax.set_xlabel("MJD (first detection in visit)")
    ax.set_ylabel("Band")
    ax.set_title(f"{title}  ({n_obj} objects)")

    ncol = max(1, int(np.ceil(n_obj / 12)))
    ax.legend(loc="upper center", bbox_to_anchor=(0.5, -0.18), ncol=ncol, fontsize=6, frameon=False)
    plt.tight_layout(rect=[0, 0.12, 1, 1])
    plt.show()


# Plot diaSources timeline
plot_all_objects_timeline(df_src, title="DIA Objects (diaSources) — temporal sampling")

# Plot forced photometry timeline (if available)
if not df_fp.empty:
    plot_all_objects_timeline(df_fp, title="DIA Objects (Forced Photometry) — temporal sampling")

## 4. Single-object inspection

In [ ]:
def plot_single_object(
    df_src: pd.DataFrame,
    dia_object_id,
    df_fp: pd.DataFrame | None = None,
) -> None:
    """
    Plot the temporal sampling by band for one DIA object.
    Overlays diaSources (filled circles) and forced photometry (crosses) if provided.

    Parameters
    ----------
    df_src : diaSources visit index DataFrame.
    dia_object_id : The diaObjectId to plot.
    df_fp : Optional forced photometry visit index DataFrame.
    """
    sub_src = df_src[df_src["diaObjectId"] == dia_object_id].dropna(subset=["mjd_first"])
    sub_fp = (
        df_fp[df_fp["diaObjectId"] == dia_object_id].dropna(subset=["mjd_first"])
        if df_fp is not None and not df_fp.empty
        else pd.DataFrame()
    )

    if sub_src.empty and sub_fp.empty:
        print(f"Object {dia_object_id} not found in visit index.")
        return

    group = sub_src["group"].iloc[0] if not sub_src.empty else "unknown"
    fig, ax = plt.subplots(figsize=(12, 4))

    # diaSources: filled circles per band
    for band, grp in sub_src.groupby("band"):
        y = [BAND_MAP.get(band, -1)] * len(grp)
        ax.scatter(
            grp["mjd_first"],
            y,
            color=BAND_COLORS.get(band, "gray"),
            label=f"{band} src",
            s=50,
            edgecolor="black",
            linewidth=0.5,
            alpha=0.9,
            marker="o",
        )

    # Forced photometry: crosses per band
    for band, grp in sub_fp.groupby("band") if not sub_fp.empty else {}:
        y = [BAND_MAP.get(band, -1)] * len(grp)
        ax.scatter(
            grp["mjd_first"],
            y,
            color=BAND_COLORS.get(band, "gray"),
            label=f"{band} fp",
            s=60,
            marker="x",
            linewidth=1.5,
            alpha=0.7,
        )

    ax.set_yticks(list(BAND_MAP.values()))
    ax.set_yticklabels(list(BAND_MAP.keys()))
    ax.set_xlabel("MJD (first detection in visit)")
    ax.set_ylabel("Band")
    ax.set_title(f"DIA Object {dia_object_id}  |  group: {group}  — temporal sampling by band")
    ax.legend(loc="upper right", frameon=False, fontsize=8, ncol=2)
    plt.tight_layout()
    plt.show()


print("Single-object plot function defined.")

In [ ]:
# ── Plot a sample of individual objects ───────────────────────────────────────
if not df_src.empty:
    sample_ids = df_src["diaObjectId"].unique()[:5]  # first 5 for quick check
    for obj_id in sample_ids:
        print(f"\nDIA Object: {obj_id}")
        plot_single_object(df_src, obj_id, df_fp=df_fp if not df_fp.empty else None)

## 5. Extract (visit, detector) list for a given DIA object

This is the core function for preparing a `pipetask run` submission:
given a `diaObjectId`, return the list of `(visit, detector)` tuples
that contain at least one detection of that object.

Both diaSources and forced photometry can be queried independently or combined.

In [ ]:
def get_visit_detector_list(
    dia_object_id,
    df_src: pd.DataFrame,
    df_fp: pd.DataFrame | None = None,
    bands: list[str] | None = None,
    source: str = "both",
    verbose: bool = True,
) -> pd.DataFrame:
    """
    Extract the list of (visit, detector) tuples for one DIA object.

    Parameters
    ----------
    dia_object_id :
        The diaObjectId to query.
    df_src : pd.DataFrame
        diaSources visit index (from visit_index.csv).
    df_fp : pd.DataFrame or None
        Forced photometry visit index (from visit_index_fp.csv). Optional.
    bands : list of str or None
        Restrict to specific bands (e.g. ['r', 'g']). None = all bands.
    source : str
        Which source to query: 'src', 'fp', or 'both'. Default 'both'.
    verbose : bool
        Print a summary table.

    Returns
    -------
    pd.DataFrame
        Unique (visit, detector, band, source_type, mjd_first, n_det,
        x_mean, y_mean, xErr_mean, yErr_mean) tuples, sorted by visit.
        Returns empty DataFrame if object not found.
    """
    frames = []

    # ── Collect from diaSources ───────────────────────────────────────────────
    if source in ("src", "both") and not df_src.empty:
        sub = df_src[df_src["diaObjectId"] == dia_object_id].copy()
        if not sub.empty:
            frames.append(sub)

    # ── Collect from forced photometry ────────────────────────────────────────
    if source in ("fp", "both") and df_fp is not None and not df_fp.empty:
        sub_fp = df_fp[df_fp["diaObjectId"] == dia_object_id].copy()
        if not sub_fp.empty:
            frames.append(sub_fp)

    if not frames:
        if verbose:
            print(f"Object {dia_object_id} not found in visit index.")
        return pd.DataFrame()

    df_all = pd.concat(frames, ignore_index=True)

    # ── Filter by band if requested ───────────────────────────────────────────
    if bands is not None:
        df_all = df_all[df_all["band"].isin(bands)]

    # ── Drop rows without visit information ───────────────────────────────────
    df_all = df_all.dropna(subset=["visit"])

    if df_all.empty:
        if verbose:
            print(f"No visit data for object {dia_object_id} (check r:visit availability).")
        return pd.DataFrame()

    # ── Build output columns ──────────────────────────────────────────────────
    keep_cols = [
        "visit",
        "detector",
        "band",
        "source_type",
        "mjd_first",
        "n_det",
        "x_mean",
        "y_mean",
        "xErr_mean",
        "yErr_mean",
        "psfFlux_mean",
    ]
    keep_cols = [c for c in keep_cols if c in df_all.columns]

    df_out = (
        df_all[keep_cols]
        .drop_duplicates(
            subset=[c for c in ["visit", "detector", "band", "source_type"] if c in df_all.columns]
        )
        .sort_values(["visit", "detector", "band"])
        .reset_index(drop=True)
    )

    if verbose:
        group = (
            df_src[df_src["diaObjectId"] == dia_object_id]["group"].iloc[0]
            if not df_src.empty and (df_src["diaObjectId"] == dia_object_id).any()
            else "unknown"
        )
        print(f"\nDIA Object: {dia_object_id}   group: {group}")
        print(f"  {len(df_out)} unique (visit, detector, band) tuples")
        print(f"  visits    : {sorted(df_out['visit'].dropna().unique().tolist())}")
        print(f"  detectors : {sorted(df_out['detector'].dropna().unique().tolist())}")
        print(f"  bands     : {sorted(df_out['band'].dropna().unique().tolist())}")
        print(df_out.to_string(index=False))

    return df_out


print("get_visit_detector_list() defined.")

## 6. Example: extract (visit, detector) for a specific object

In [ ]:
# ── Select target object ──────────────────────────────────────────────────────
# Change this to any diaObjectId present in the visit index.
if not df_src.empty:
    TARGET_OID = df_src["diaObjectId"].iloc[0]  # first object as default
else:
    TARGET_OID = None

print(f"Target object: {TARGET_OID}")

if TARGET_OID is not None:
    # Plot the object's temporal sampling
    plot_single_object(df_src, TARGET_OID, df_fp=df_fp if not df_fp.empty else None)

    # Extract (visit, detector) from diaSources only
    print("\n--- diaSources only ---")
    df_vd_src = get_visit_detector_list(TARGET_OID, df_src, df_fp=None, source="src")

    # Extract (visit, detector) from forced photometry only (if available)
    if not df_fp.empty:
        print("\n--- Forced photometry only ---")
        df_vd_fp = get_visit_detector_list(TARGET_OID, df_src, df_fp=df_fp, source="fp")

    # Extract combined (visit, detector) from both
    print("\n--- Combined (src + fp) ---")
    df_vd_all = get_visit_detector_list(
        TARGET_OID, df_src, df_fp=df_fp if not df_fp.empty else None, source="both"
    )

## 7. Batch extraction for a group or band selection

In [ ]:
def get_visit_detector_for_group(
    df_src: pd.DataFrame,
    df_fp: pd.DataFrame | None = None,
    groups: list[str] | None = None,
    bands: list[str] | None = None,
    source: str = "src",
) -> pd.DataFrame:
    """
    Extract all unique (visit, detector) pairs for a set of groups and/or bands.

    Parameters
    ----------
    df_src : diaSources visit index.
    df_fp : Forced photometry visit index. Optional.
    groups : List of group names to include. None = all groups.
    bands : List of bands to include. None = all bands.
    source : 'src', 'fp', or 'both'.

    Returns
    -------
    pd.DataFrame
        Unique (visit, detector) pairs with band and group information.
    """
    frames = []
    if source in ("src", "both") and not df_src.empty:
        frames.append(df_src)
    if source in ("fp", "both") and df_fp is not None and not df_fp.empty:
        frames.append(df_fp)

    if not frames:
        return pd.DataFrame()

    df = pd.concat(frames, ignore_index=True)

    if groups is not None:
        df = df[df["group"].isin(groups)]
    if bands is not None:
        df = df[df["band"].isin(bands)]

    df = df.dropna(subset=["visit"])

    keep = [
        c
        for c in ["visit", "detector", "band", "group", "source_type", "diaObjectId", "mjd_first"]
        if c in df.columns
    ]
    return (
        df[keep]
        .drop_duplicates(subset=[c for c in ["visit", "detector", "band", "source_type"] if c in df.columns])
        .sort_values(["visit", "detector"])
        .reset_index(drop=True)
    )


# ── Example: Gaia stable stars, r-band, diaSources ───────────────────────────
if not df_src.empty:
    df_drp_gaia_r = get_visit_detector_for_group(
        df_src,
        df_fp=df_fp if not df_fp.empty else None,
        groups=["gaia_star_stable_hq", "gaia_star_stable"],
        bands=["r"],
        source="src",
    )
    print(f"Gaia stable stars, r-band — {len(df_drp_gaia_r)} unique (visit, detector) pairs")
    print(df_drp_gaia_r.head(20).to_string())

## 8. Export CSV for pipetask run

In [ ]:
def export_pipetask_csv(
    df_vd: pd.DataFrame,
    output_name: str,
    dir_data: str = DIR_DATA,
) -> str:
    """
    Save a (visit, detector) DataFrame to CSV for use with pipetask run.

    The output CSV contains at minimum: visit, detector, band.
    It also prints a ready-to-use Butler WHERE clause.

    Parameters
    ----------
    df_vd : DataFrame with columns visit, detector (at minimum).
    output_name : Base name for the output file (without .csv).
    dir_data : Directory to save the file.

    Returns
    -------
    str : Path to the saved CSV file.
    """
    if df_vd.empty:
        print("Empty DataFrame — nothing to save.")
        return ""

    path = os.path.join(dir_data, f"{output_name}.csv")
    df_vd.to_csv(path, index=False)
    print(f"Saved: {path}  ({len(df_vd)} rows)")

    # Print Butler WHERE clause
    visits = sorted(df_vd["visit"].dropna().unique().astype(int).tolist())
    detectors = sorted(df_vd["detector"].dropna().unique().astype(int).tolist())
    visit_str = ", ".join(str(v) for v in visits)
    det_str = ", ".join(str(d) for d in detectors)
    print(f"\n  Butler WHERE clause:")
    print(f'  "visit IN ({visit_str}) AND detector IN ({det_str})"')
    print(f"\n  Total visits    : {len(visits)}")
    print(f"  Total detectors : {len(detectors)}")

    return path


# ── Export example for a single target object ─────────────────────────────────
if TARGET_OID is not None and not df_vd_all.empty:
    export_pipetask_csv(
        df_vd_all,
        output_name=f"pipetask_vd_{TARGET_OID}",
    )

# ── Export example for Gaia stable stars r-band ───────────────────────────────
if not df_src.empty and not df_drp_gaia_r.empty:
    export_pipetask_csv(
        df_drp_gaia_r,
        output_name="pipetask_vd_gaia_stable_r",
    )

## 9. Butler snippet — ready to use at USDF

In [ ]:
# ── Print a copy-pasteable Butler snippet ─────────────────────────────────────
butler_snippet = """
# ── Butler Rubin snippet (USDF) ───────────────────────────────────────────────
# Adapt repo, collection, and WHERE clause to your setup.

import pandas as pd
from lsst.daf.butler import Butler

# Load the (visit, detector) CSV produced by this notebook
df_vd = pd.read_csv("data_FINK_BLOCK_LC_01/pipetask_vd_gaia_stable_r.csv")
visits    = sorted(df_vd["visit"].dropna().unique().astype(int))
detectors = sorted(df_vd["detector"].dropna().unique().astype(int))

# Initialise the Butler
repo   = "/repo/main"
coll   = "LSSTCam/runs/DRP/..."
butler = Butler(repo, collections=[coll])

# Option A: iterate over (visit, detector) pairs from the DataFrame
for _, row in df_vd[["visit", "detector"]].drop_duplicates().iterrows():
    try:
        calexp = butler.get("calexp", visit=int(row["visit"]), detector=int(row["detector"]))
        print(f"visit={row['visit']}  det={row['detector']}  OK  shape={calexp.image.array.shape}")
    except Exception as e:
        print(f"visit={row['visit']}  det={row['detector']}  ERROR: {e}")

# Option B: build a WHERE clause for pipetask run
visit_str = ", ".join(str(v) for v in visits)
det_str   = ", ".join(str(d) for d in detectors)
where_clause = f"visit IN ({visit_str}) AND detector IN ({det_str})"
print(f"WHERE: {where_clause}")

# Equivalent shell command:
# pipetask run \\
#     -p ${DRP_PIPE_DIR}/pipelines/LSSTCam/DRP.yaml#stage1-single-visit,stage2-recalibrate \
#     -b /repo/main \\
#     -i LSSTCam/default \\
#     -o u/username/fink_calib_run01 \\
#     -d "{where_clause}" \\
#     --register-dataset-types
# ─────────────────────────────────────────────────────────────────────────────
"""
print(butler_snippet)

## 10. Summary statistics of the visit index

In [ ]:
for label, df in (("diaSources", df_src), ("Forced Photometry", df_fp)):
    if df.empty:
        print(f"\n{label}: not available")
        continue
    print(f"\n{'=' * 60}")
    print(f"  {label} visit index summary")
    print(f"{'=' * 60}")
    print(f"  Total rows               : {len(df)}")
    print(f"  Unique diaObjectId       : {df['diaObjectId'].nunique()}")
    print(f"  Unique visits            : {df['visit'].dropna().nunique()}")
    print(f"  Unique (visit, detector) : {df[['visit', 'detector']].drop_duplicates().dropna().shape[0]}")
    print(f"  Bands represented        : {sorted(df['band'].dropna().unique())}")

    if "group" in df.columns:
        print(f"\n  Objects by group:")
        print(
            df.groupby("group")["diaObjectId"]
            .nunique()
            .sort_values(ascending=False)
            .rename("n_objects")
            .to_string()
        )

    print(f"\n  Visits per band:")
    print(df.groupby("band")["visit"].nunique().rename("n_unique_visits").to_string())

    if "xErr_mean" in df.columns:
        xe = df["xErr_mean"].dropna()
        print(f"\n  xErr_mean : {xe.mean():.3f} ± {xe.std():.3f} px  (N={len(xe)})")
        ye = df["yErr_mean"].dropna()
        print(f"  yErr_mean : {ye.mean():.3f} ± {ye.std():.3f} px  (N={len(ye)})")